# 🍏 Health Resource Search Agent Tutorial 🍎

Welcome to the **Health Resource Search Agent** tutorial! We'll use **Azure AI Foundry** SDKs to build an assistant that can:

1. **Upload** health and recipe files into a vector store.
2. **Create an Agent** with a **File Search** tool.
3. **Search** these documents for relevant dietary info.
4. **Answer** health and wellness questions (with disclaimers!).

### ⚠️ Important Medical Disclaimer ⚠️
> **All health information in this notebook is for general educational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment.** Always seek the advice of a qualified healthcare professional with any questions you may have.

## Prerequisites
- Complete Agent basics notebook - [1-basics.ipynb](1-basics.ipynb)
- **Roles**  
  1. **Azure AI Developer** on your Azure AI Foundry project.
  2. **Storage Blob Data Contributor** on the project’s Storage account.
  3. If standard agent setup is used with your own Search resource, also ensure you have **Cognitive Search Data Contributor** on that resource.

## Let's Get Searching!
We'll show you how to upload some sample files, create a vector store for them, then spin up an agent that can search these resources for dietary guidelines, recipes, and more. Enjoy!

<img src="./seq-diagrams/3-file-search.png" width="30%"/>


## 1. Initial Setup
Here we import needed libraries, load environment variables from `.env`, and initialize our **AIProjectClient**. Let's do this! 🎉

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables from parent .env
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Initialize the endpoint-based AIProjectClient and its OpenAI (Responses API) client.
# The OpenAI client handles files, vector stores, conversations and responses.
try:
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully initialized AIProjectClient + OpenAI client")
except Exception as e:
    print(f"❌ Error initializing project client: {e}")

## 2. Prepare Sample Files 🍲🗒
We'll create some dummy .md files (for recipes and guidelines). Then we'll store them in a vector store for searching.


In [ ]:
def create_sample_files():
    recipes_md = (
        """# Healthy Recipes Database\n\n"
        "## Gluten-Free Recipes\n"
        "1. Quinoa Bowl\n"
        "   - Ingredients: quinoa, vegetables, olive oil\n"
        "   - Instructions: Cook quinoa, add vegetables\n\n"
        "2. Rice Pasta with Vegetables\n"
        "   - Ingredients: rice pasta, mixed vegetables\n"
        "   - Instructions: Boil pasta, sauté vegetables\n\n"
        "## Diabetic-Friendly Recipes\n"
        "1. Low-Carb Stir Fry\n"
        "   - Ingredients: chicken, vegetables, tamari sauce\n"
        "   - Instructions: Cook chicken, add vegetables\n\n"
        "2. Greek Salad\n"
        "   - Ingredients: cucumber, tomatoes, feta, olives\n"
        "   - Instructions: Chop vegetables, combine\n\n"
        "## Heart-Healthy Recipes\n"
        "1. Baked Salmon\n"
        "   - Ingredients: salmon, lemon, herbs\n"
        "   - Instructions: Season salmon, bake\n\n"
        "2. Mediterranean Bowl\n"
        "   - Ingredients: chickpeas, vegetables, tahini\n"
        "   - Instructions: Combine ingredients\n"""
    )

    guidelines_md = (
        """# Dietary Guidelines\n\n"
        "## General Guidelines\n"
        "- Eat a variety of foods\n"
        "- Control portion sizes\n"
        "- Stay hydrated\n\n"
        "## Special Diets\n"
        "1. Gluten-Free Diet\n"
        "   - Avoid wheat, barley, rye\n"
        "   - Focus on naturally gluten-free foods\n\n"
        "2. Diabetic Diet\n"
        "   - Monitor carbohydrate intake\n"
        "   - Choose low glycemic foods\n\n"
        "3. Heart-Healthy Diet\n"
        "   - Limit saturated fats\n"
        "   - Choose lean proteins\n"""
    )

    # Save to local .md files
    with open("recipes.md", "w", encoding="utf-8") as f:
        f.write(recipes_md)
    with open("guidelines.md", "w", encoding="utf-8") as f:
        f.write(guidelines_md)

    print("📄 Created sample resource files: recipes.md, guidelines.md")
    return ["recipes.md", "guidelines.md"]

sample_files = create_sample_files()

#### ✨ Note on Search Permissions
When creating the vector store, you must also have **Cognitive Search Data Contributor** role on your Azure AI Search resource (if you're using the standard agent setup with your own Search resource). Missing this role will often cause a **Forbidden** error. See [Authentication Setup](../../1-introduction/1-authentication.ipynb#4-add-agent-service-permissions) for details on configuring permissions.


## 3. Create a Vector Store 📚
We'll upload our newly created files and group them into a single vector store for searching. This is how the agent can later find relevant text.

In [ ]:
def create_vector_store(files, store_name="my_health_resources"):
    try:
        # Step 1: Create an (empty) vector store via the OpenAI client
        # A vector store converts text into numerical vectors for semantic search
        vs = openai_client.vector_stores.create(name=store_name)
        print(f"🎉 Created vector store '{store_name}', ID: {vs.id}")

        # Step 2: Upload each file, then attach it to the vector store.
        # create_and_poll waits until ingestion (parse + chunk + embed) is complete.
        uploaded_ids = []
        for fp in files:
            with open(fp, "rb") as fh:
                uploaded = openai_client.files.create(purpose="assistants", file=fh)
            openai_client.vector_stores.files.create_and_poll(
                vector_store_id=vs.id,
                file_id=uploaded.id,
            )
            uploaded_ids.append(uploaded.id)
            print(f"✅ Uploaded & indexed: {fp} -> File ID: {uploaded.id}")

        return vs, uploaded_ids
    except Exception as e:
        print(f"❌ Error creating vector store: {e}")
        return None, []

# Initialize empty variables to store our vector store and file IDs
vector_store, file_ids = None, []

# If we successfully created sample files earlier, create a vector store from them
if sample_files:
    vector_store, file_ids = create_vector_store(sample_files, "health_resources_example")

## 4. Create the Health Resource Agent 🔎
We use a **FileSearchTool** pointing to our newly created vector store, then create the Agent with instructions about disclaimers, dietary help, etc.

In [ ]:
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition

def create_health_resource_agent(vstore_id):
    try:
        # Create an agent version with the FileSearchTool bound to our vector store.
        # The agent can find relevant content even if the exact words don't match.
        agent = project_client.agents.create_version(
            agent_name="health-search-agent",
            definition=PromptAgentDefinition(
                # Specify which model to use - fallback to gpt-5.4 if not set
                model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
                # These instructions act like a personality and rule set for our agent
                instructions="""
                    You are a health resource advisor with access to dietary and recipe files.
                    You:
                    1. Always present disclaimers (you're not a doctor!)
                    2. Provide references to the files when possible
                    3. Focus on general nutrition or recipe tips.
                    4. Encourage professional consultation for more detailed advice.
                """,
                tools=[FileSearchTool(vector_store_ids=[vstore_id])],
            ),
            description="Health resource search agent over uploaded documents.",
        )
        print(f"🎉 Created health resource agent '{agent.name}', version: {agent.version}")
        return agent
    except Exception as e:
        print(f"❌ Error creating health resource agent: {e}")
        return None

# Initialize our agent variable
health_agent = None

# Only create the agent if we successfully created a vector store earlier
if vector_store:
    health_agent = create_health_resource_agent(vector_store.id)

## 5. Searching Health Resources 🏋️👩‍🍳
We'll open a new conversation and ask queries like "Gluten-free recipe ideas?" or "Heart-healthy meal plan?" The agent will run file search over the vector store to find relevant info. We store each `(query, response)` pair for the next step.

In [ ]:
search_responses = []

def ask_search_question(agent, conversation_id, user_question):
    """Send a query to the file-search agent within a conversation, via the Responses API."""
    try:
        print(f"🔎 Searching: '{user_question}'")
        # A single responses.create call runs the agent, which uses file search
        # over the vector store, and stores the result in the conversation.
        response = openai_client.responses.create(
            conversation=conversation_id,
            input=user_question,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        print(f"🤖 Response status: {response.status}")
        return response
    except Exception as e:
        print(f"❌ Error searching question: {e}")
        return None

# Now let's test our search functionality!
search_conversation = None
if health_agent:
    # Create a new conversation to hold the Q&A
    search_conversation = openai_client.conversations.create()
    print(f"📝 Created new search conversation, ID: {search_conversation.id}")

    # Define some test questions that demonstrate different types of health queries
    queries = [
        "Could you suggest a gluten-free lunch recipe?",
        "Show me some heart-healthy meal ideas.",
        "What guidelines do you have for someone with diabetes?"
    ]

    # Process each query; the conversation maintains context between questions
    for q in queries:
        resp = ask_search_question(health_agent, search_conversation.id, q)
        if resp:
            search_responses.append((q, resp))

## 6. View Results & Citations 📄
We'll print each answer and check the response annotations to see whether the agent cited the correct files.

In [ ]:
def display_search_results(pairs):
    """Print each answer and any file citations from the response annotations."""
    try:
        print("\n🗣️ Conversation results:")
        for user_question, response in pairs:
            print(f"\nUSER: {user_question}")
            print(f"ASSISTANT: {response.output_text}\n")

            # The agent cites source documents as file_citation annotations
            # on the output_text content. Let's surface them.
            citations = []
            for item in (response.output or []):
                if getattr(item, "type", "") != "message":
                    continue
                for content in (getattr(item, "content", None) or []):
                    for ann in (getattr(content, "annotations", None) or []):
                        if getattr(ann, "type", "") == "file_citation":
                            citations.append(getattr(ann, "filename", "") or getattr(ann, "file_id", ""))
            if citations:
                print("📎 Citations:")
                for c in citations:
                    print(f"   - {c}")
    except Exception as e:
        print(f"❌ Error displaying messages: {e}")

# Display the results for our search conversation
if search_responses:
    display_search_results(search_responses)

## 7. Cleanup & Best Practices 🧹
We'll optionally remove the vector store, the uploaded files, and the agent. In a production environment, you might keep them around longer. Meanwhile, here are some tips:

1. **Resource Management**
   - Keep files grouped by category, regularly prune old or irrelevant files.
   - Clear out test agents or vector stores once you're done.

2. **Search Queries**
   - Provide precise or multi-part queries.
   - Consider synonyms or alternative keywords ("gluten-free" vs "celiac").
   
3. **Health Information**
   - Always disclaim that you are not a medical professional.
   - Encourage users to see doctors for specific diagnoses.

4. **Performance**
   - Keep an eye on vector store size.
   - Evaluate search accuracy with `azure-ai-evaluation`!


In [ ]:
def cleanup_all():
    try:
        # Delete the AI agent version we created
        if 'health_agent' in globals() and health_agent:
            project_client.agents.delete_version(
                agent_name=health_agent.name,
                agent_version=health_agent.version,
            )
            print("🗑️ Deleted health resource agent.")

        # Delete the vector store (removes the file associations/embeddings)
        if 'vector_store' in globals() and vector_store:
            openai_client.vector_stores.delete(vector_store.id)
            print("🗑️ Deleted vector store.")

        # Remove the underlying uploaded file objects from the service
        if 'file_ids' in globals() and file_ids:
            for fid in file_ids:
                openai_client.files.delete(fid)
            print("🗑️ Deleted uploaded files from the service.")

        # Clean up any local files we created during the demo
        if 'sample_files' in globals() and sample_files:
            for sf in sample_files:
                if os.path.exists(sf):
                    os.remove(sf)
            print("🗑️ Deleted local sample files.")

    except Exception as e:
        # If anything goes wrong during cleanup, we'll see what happened
        print(f"❌ Error during cleanup: {e}")

# Run our cleanup function to remove all resources we created
cleanup_all()

# Congratulations! 🎉
You've created a **Health Resource Search Agent** that:
1. Uses a **Vector Store** to store sample recipes & guidelines.
2. **Searches** them to answer queries.
3. **Provides disclaimers** reminding users to consult professionals.

Feel free to adapt this approach for your own corporate documents, product manuals, or custom health resources.

Happy Searching! 🎉

#### Let's proceed to [4-bing_grounding.ipynb](4-bing_grounding.ipynb)